# 04 Statistical Analysis — Inference on Startup Failure

## Objective
Validate EDA patterns with formal statistical tests: association tests, group-difference tests, correlations, and logistic regression.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, pointbiserialr
import statsmodels.api as sm

pd.set_option('display.max_columns', None)

In [2]:
# Robust project-root detection
cwd = Path.cwd().resolve()
if (cwd / 'data' / 'processed' / 'startups_cleaned.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data' / 'processed' / 'startups_cleaned.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Could not locate data/processed/startups_cleaned.csv from current working directory')

CLEAN_PATH = PROJECT_ROOT / 'data' / 'processed' / 'startups_cleaned.csv'
df = pd.read_csv(CLEAN_PATH, low_memory=False)

for dcol in ['founded_at', 'first_funding_at', 'last_funding_at']:
    if dcol in df.columns:
        df[dcol] = pd.to_datetime(df[dcol], errors='coerce')

if 'is_closed' not in df.columns:
    df['is_closed'] = (df['status'].astype(str).str.lower() == 'closed').astype(int)
if 'reached_series_a' not in df.columns:
    df['reached_series_a'] = (df.get('round_a', 0).fillna(0) > 0).astype(int)
if 'days_to_first_funding' not in df.columns:
    first_i8 = pd.to_datetime(df['first_funding_at'], errors='coerce').to_numpy(dtype='datetime64[ns]').astype('int64')
    founded_i8 = pd.to_datetime(df['founded_at'], errors='coerce').to_numpy(dtype='datetime64[ns]').astype('int64')
    ns_per_day = 86_400_000_000_000
    nat = np.iinfo('int64').min
    days = (first_i8 - founded_i8) / ns_per_day
    df['days_to_first_funding'] = np.where((first_i8 == nat) | (founded_i8 == nat), np.nan, days)
if 'avg_funding_per_round' not in df.columns:
    df['avg_funding_per_round'] = np.where(
        (df['funding_rounds'] > 0) & (df['funding_total_usd'] > 0),
        df['funding_total_usd'] / df['funding_rounds'],
        np.nan
    )
if 'is_usa' not in df.columns:
    df['is_usa'] = (df['country_code'] == 'USA').astype(int)

print(f'Dataset for inference: {len(df):,} rows')
print('Using file:', CLEAN_PATH)

Dataset for inference: 49,437 rows
Using file: /Users/rashmianand/Desktop/SectionC_G17_WhyStartupsFail/data/processed/startups_cleaned.csv


## 1) Chi-Square Test: Status vs Reached Series A

In [3]:
ct = pd.crosstab(df['status'].fillna('unknown'), df['reached_series_a'])
chi2, p_value, dof, expected = chi2_contingency(ct)
n = ct.values.sum()
r, k = ct.shape
cramers_v = np.sqrt((chi2 / n) / max(1, min(r - 1, k - 1)))

chi_result = pd.DataFrame({
    'metric': ['chi2', 'p_value', 'dof', 'cramers_v'],
    'value': [chi2, p_value, dof, cramers_v]
})

print('Contingency Table:')
print(ct)
chi_result

Contingency Table:
reached_series_a      0     1
status                       
acquired           2480  1212
closed             2144   459
operating         34693  7134
unknown            1117   198


,metric,value
0,chi2,5.764106e+02
1,p_value,1.309426e-124
2,dof,3.000000e+00
3,cramers_v,1.079792e-01


**Finding + Context (Chi-Square)**
- If `p_value < 0.05`, startup outcome status and Series A progression are statistically associated.
- In business terms, this means stage progression and eventual outcome are connected, not independent processes.
- Real-world interpretation: companies that cannot cross early financing checkpoints often face restricted hiring, weaker go-to-market execution, and higher shutdown probability.
- `Cramér's V` should be read as practical strength: significant p-values can still have small-to-moderate real-world effect sizes.

## 2) Mann-Whitney U Test: Funding of Closed vs Operating

In [4]:
closed_f = df[(df['status'] == 'closed') & (df['funding_total_usd'] > 0)]['funding_total_usd'].dropna()
oper_f = df[(df['status'] == 'operating') & (df['funding_total_usd'] > 0)]['funding_total_usd'].dropna()

u_stat, p_u = mannwhitneyu(closed_f, oper_f, alternative='two-sided')

mw_result = pd.DataFrame({
    'group': ['closed', 'operating'],
    'n': [len(closed_f), len(oper_f)],
    'median_funding_usd': [closed_f.median(), oper_f.median()]
})

print(f'Mann-Whitney U: U={u_stat:.2f}, p={p_u:.4g}')
mw_result

Mann-Whitney U: U=32474908.00, p=1.014e-22


,group,n,median_funding_usd
0,closed,2158,1000000.0
1,operating,34424,1757977.5


**Finding + Context (Mann-Whitney U)**
- This test checks whether the funding distribution of `closed` startups differs from `operating` startups without assuming normality.
- When closed startups have lower median funding, the likely explanation is reduced runway: less time to iterate product, acquire customers, and survive slow revenue periods.
- Real-world example: an early-stage firm with limited funding may fail before reaching stable demand, while a better-capitalized peer can survive long enough to pivot.

## 3) Kruskal-Wallis Test: Funding Rounds Across Status Groups

In [5]:
status_groups = ['closed', 'operating', 'acquired', 'ipo']
groups = [df[df['status'] == s]['funding_rounds'].dropna() for s in status_groups]
groups = [g for g in groups if len(g) > 0]

h_stat, p_kw = kruskal(*groups)

print(f'Kruskal-Wallis: H={h_stat:.2f}, p={p_kw:.4g}')

df.groupby('status')['funding_rounds'].agg(['count', 'median', 'mean']).sort_values('count', ascending=False)

Kruskal-Wallis: H=519.52, p=1.541e-113


,count,median,mean
status,,,
operating,41827,1.0,1.689507
acquired,3692,2.0,2.013814
closed,2603,1.0,1.434114


**Finding + Context (Kruskal-Wallis)**
- A significant result here means funding-round depth differs across status groups, reinforcing that startup outcomes evolve through financing stages.
- Causal story in practice: each extra validated round often reflects investor confidence, milestone achievement, and better operating resilience.
- This does not mean “more rounds automatically guarantee success,” but it does indicate materially different lifecycle pathways.

## 4) Point-Biserial Correlation with Closure

In [6]:
corr_targets = ['funding_total_usd', 'funding_rounds', 'avg_funding_per_round', 'days_to_first_funding']
rows = []
for c in corr_targets:
    sub = df[['is_closed', c]].dropna()
    if len(sub) < 30:
        continue
    r, p = pointbiserialr(sub['is_closed'], sub[c])
    rows.append({'feature': c, 'n': len(sub), 'r': r, 'p_value': p})

pb_result = pd.DataFrame(rows).sort_values('p_value')
pb_result

,feature,n,r,p_value
1,funding_rounds,48122,-0.049079,4.636180e-27
3,days_to_first_funding,37628,-0.050900,5.098394e-23
2,avg_funding_per_round,39800,-0.013975,5.302679e-03
0,funding_total_usd,39800,-0.010537,3.553716e-02


**Finding + Context (Point-Biserial Correlation)**
- These correlations show direction and strength between closure and each numeric feature.
- Even if coefficients are small, consistent direction across multiple variables is informative in complex systems.
- Real-world meaning: startup failure is rarely triggered by one metric; multiple weak signals can still combine into strong overall risk.

## 5) Logistic Regression: Independent Predictors of Closure

In [7]:
model_df = df[['is_closed', 'funding_rounds', 'funding_total_usd', 'reached_series_a', 'days_to_first_funding', 'is_usa']].copy()
model_df = model_df.dropna()
model_df = model_df[model_df['funding_total_usd'] > 0]
model_df['log_funding_total_usd'] = np.log1p(model_df['funding_total_usd'])

X = model_df[['funding_rounds', 'log_funding_total_usd', 'reached_series_a', 'days_to_first_funding', 'is_usa']]
X = sm.add_constant(X)
y = model_df['is_closed']

logit = sm.Logit(y, X).fit(disp=False)
coef = logit.params
odds_ratio = np.exp(coef)
pvals = logit.pvalues

logit_summary = pd.DataFrame({
    'coef': coef,
    'odds_ratio': odds_ratio,
    'p_value': pvals
}).sort_values('p_value')

print(f'Pseudo R^2: {logit.prsquared:.4f}')
print(f'N used in regression: {len(model_df):,}')
logit_summary

Pseudo R^2: 0.0313
N used in regression: 31,437


,coef,odds_ratio,p_value
funding_rounds,-0.268977,0.764161,9.189819e-19
days_to_first_funding,-0.000166,0.999834,6.070686e-15
log_funding_total_usd,-0.092587,0.911570,2.853156e-11
const,-1.055032,0.348181,2.109679e-10
reached_series_a,0.187952,1.206776,9.622476e-03
is_usa,0.088757,1.092815,9.149882e-02


**Finding + Context (Logistic Regression)**
- Logistic regression estimates each predictor’s independent relationship with closure while controlling for other variables.
- This is closer to real decision-making than single-variable views because investors and operators evaluate multiple signals together.
- Real-world example: a startup with moderate funding may still have lower risk if it progressed through key stages and maintained faster early execution.
- Use odds ratios for practical interpretation: values below 1 indicate lower closure odds as that predictor increases.

## Statistical Conclusion
- Statistical significance tells us the relationships are unlikely to be random; effect sizes tell us how strong they are in practical terms.
- The consistent pattern across tests suggests failure risk is linked to capital depth, financing trajectory, and early execution speed.
- The weak-to-moderate magnitudes also confirm a real-world truth: startup failure is multi-factorial and context-dependent, not explained by one metric alone.
- Best project takeaway: combine quantitative signals with sector and cohort context for better risk interpretation.